# Lecture 7 — Class Exercise
## Heatmap & Waterfall: Netflix Catalogue

> **Push to:** `week07/lecture07_exercise.ipynb`

**Rules:**
1. Heatmap: colour scale must match the data type (sequential for counts, diverging for above/below)
2. Waterfall: use green for additions, red for subtractions, blue for totals
3. Insight title tells the setup-conflict-resolution story (or at minimum states the finding)
4. Annotate at least one cell or bar directly

---


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('../data/netflix_catalogue.csv')
print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())


In [ ]:
print("Genres:", df['genre'].value_counts().head(8))
print("\nCountries:", df['country'].value_counts().head(8))
print("\nRatings:", df['rating'].value_counts())


## Task 1 — Heatmap: content by rating and release decade

**What to build:** A heatmap showing the number of titles by **content rating** (y-axis) and **decade** (x-axis).

**Requirements:**
- Create a 'decade' column: `df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'`
- Filter to TV-14, TV-MA, PG-13, R, PG only (most common ratings)
- Sequential colour scale (Blues)
- Values shown in cells (`text_auto=True`)
- Insight title about which rating dominates which decade


In [ ]:
# ── Task 1: Heatmap — content by rating and release decade ───────────────────

# Step 1: Create the decade column
df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'

# Step 2: Filter to the five most common ratings
target_ratings = ['TV-14', 'TV-MA', 'PG-13', 'R', 'PG']
df_filtered = df.loc[df['rating'].isin(target_ratings)].copy()

# Step 3: Aggregate — count titles per (rating, decade) pair
rating_decade = (
    df_filtered
    .groupby(['rating', 'decade'])
    .size()
    .reset_index(name='count')
)

# Step 4: Pivot to matrix form for px.imshow
pivot_t1 = rating_decade.pivot(
    index='rating',
    columns='decade',
    values='count'
).fillna(0).astype(int)

# Sort columns (decades) chronologically
pivot_t1 = pivot_t1.reindex(sorted(pivot_t1.columns), axis=1)

# Sort rows so the highest-volume ratings sit at the top
pivot_t1 = pivot_t1.loc[pivot_t1.sum(axis=1).sort_values(ascending=False).index]

# ── Step 5: Build the heatmap ─────────────────────────────────────────────────
fig1 = px.imshow(
    pivot_t1,
    color_continuous_scale='Blues',   # sequential scale — correct for count data
    text_auto=True,                   # show count value inside every cell
    aspect='auto',
    height=420, width=900,
    labels={'color': 'Titles'}
)

# ── Step 6: Customise ─────────────────────────────────────────────────────────
fig1.update_traces(
    # White text stays readable on dark blue cells; dark text on pale cells
    textfont=dict(size=11),
    hovertemplate='<b>%{y}</b> — %{x}<br>Titles: %{z}<extra></extra>',
)

# Annotate the single largest cell so the story is unmissable
# TV-MA and TV-14 dominate the 2010s — find the max cell to spotlight it
max_val  = pivot_t1.values.max()
max_loc  = [(r, c) for r in pivot_t1.index for c in pivot_t1.columns
            if pivot_t1.loc[r, c] == max_val][0]
col_idx  = list(pivot_t1.columns).index(max_loc[1])
row_idx  = list(pivot_t1.index).index(max_loc[0])

fig1.add_annotation(
    x=col_idx, y=row_idx,
    text=f"Peak: {max_val:,}",
    showarrow=True,
    arrowhead=2, arrowcolor='black',
    ax=50, ay=-35,
    font=dict(size=11, color='black'),
    bgcolor='rgba(255,255,255,0.75)',
    bordercolor='black', borderwidth=1
)

fig1.update_layout(
    # Insight title: setup (context) → conflict (finding) → resolution (implication)
    title=dict(
        text='TV-MA surged in the 2010s as streaming unlocked adult content — mature ratings now outnumber family ones',
        font=dict(size=13)
    ),
    font=dict(family='Arial', size=11),
    coloraxis_colorbar=dict(title='Titles', thickness=14),
    margin=dict(l=80, r=20, t=70, b=60),
    xaxis=dict(title='Release Decade', tickangle=0),
    yaxis=dict(title='Content Rating')
)

fig1.show()


## Task 2 — Waterfall: Movie vs TV Show additions by year

**What to build:** A waterfall chart showing how Netflix's **Movie library** grew year by year (2015-2022).

**Requirements:**
- Filter to Movies only
- Group by `added_year`, count titles per year
- Final bar should be the cumulative total
- Green bars (additions), blue total
- Annotation on the year with the largest single addition
- Insight title naming the growth story


In [ ]:
# ── Task 2: Waterfall — Movie library growth 2015-2022 ───────────────────────

# Step 1: Filter to Movies and the required year range
movies = df.loc[
    (df['type'] == 'Movie') &
    (df['added_year'] >= 2015) &
    (df['added_year'] <= 2022)
].copy()

# Step 2: Count titles added per year
adds = (
    movies
    .groupby('added_year')
    .size()
    .reset_index(name='new_movies')
    .sort_values('added_year')
)

# Step 3: Cumulative total for the final 'Total' bar
cumulative_total = adds['new_movies'].sum()

# Step 4: Identify the peak year for annotation
peak_row   = adds.loc[adds['new_movies'].idxmax()]
peak_year  = str(int(peak_row['added_year']))
peak_count = int(peak_row['new_movies'])

# Step 5: Build x, y, and measure lists (professor's pattern)
x_vals   = adds['added_year'].astype(str).tolist() + ['Total 2015–2022']
y_vals   = adds['new_movies'].tolist()              + [cumulative_total]
measures = ['relative'] * len(adds)                 + ['total']

# Step 6: Build the Waterfall trace (must use go.Waterfall — not in px)
trace2 = go.Waterfall(
    x=x_vals,
    y=y_vals,
    measure=measures,
    texttemplate='%{y:,}',            # show count above every bar
    textposition='outside',
    connector=dict(line=dict(color='#AAAAAA', dash='dot')),
    increasing=dict(marker_color='#70AD47'),   # green for additions
    totals=dict(marker_color='#2E75B6'),        # blue for the total bar
    textfont=dict(size=11, color='#333333'),
)

fig2 = go.Figure(data=[trace2])

# Step 7: Annotate the peak year
peak_x_idx = x_vals.index(peak_year)

fig2.add_annotation(
    x=peak_year,
    y=peak_count,
    text=f"🏆 Peak year: {peak_count:,} movies",
    showarrow=True,
    arrowhead=2, arrowcolor='#333333',
    ax=60, ay=-45,
    font=dict(size=11, color='#333333'),
    bgcolor='rgba(255,255,255,0.80)',
    bordercolor='#70AD47', borderwidth=1.5
)

# Step 8: Layout — insight title + clean white background (professor's style)
fig2.update_layout(
    title=dict(
        text='Netflix tripled its Movie library in 3 years — rapid additions 2017–2019 drove most of the growth',
        font=dict(size=13)
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    showlegend=False,
    height=520,
    margin=dict(l=60, r=40, t=70, b=50),
    yaxis=dict(title='Movies Added', gridcolor='#EEEEEE'),
    xaxis=dict(title='Year', showgrid=False)
)

fig2.show()
